# Tester la performance du modèle en fonction du nombre de runs 

### On teste le même modèle avec les mêmes paramètre, mais sur 10 runs et on calcule la moyenne du test accuracy

### Imports

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "/home/f24achba/Documents/Edge AI/52-Hertz-KWS/src")))

In [2]:
import tensorflow as tf
import numpy as np
import os, glob
import json, random
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from preprocessing import AudioPipeline
from preprocessing.dataset_pipeline import load_and_preprocess, make_dataset
from preprocessing.transfer_model import build_transfer_model
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold   

I0000 00:00:1781016073.014680  112073 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781016073.040484  112073 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781016073.782664  112073 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Config

In [3]:
DATA_DIR    = "/home/f24achba/Documents/Edge AI/52-Hertz-KWS/data/raw"
N_RUNS      = 10
CLASSES     = ["talky", "silence", "unknown"]
BATCH_SIZE  = 16      
EPOCHS   = 40       # backbone gelé
LR       = 1e-3
K_FOLDS  = 5
SEED       = 42
KERAS_PATH  = "/home/f24achba/Documents/Edge AI/52-Hertz-KWS/src/models/miniresnetv2_s1_64x50_tl.keras"
pipeline = AudioPipeline("/home/f24achba/Documents/Edge AI/52-Hertz-KWS/configuration/config.yaml")

## Chargement des chemins + labels 

In [4]:
all_paths, all_labels = [], []

for label_idx, class_name in enumerate(CLASSES):
    wavs = glob.glob(os.path.join(DATA_DIR, class_name, "*.wav"))
    all_paths  += wavs
    all_labels += [label_idx] * len(wavs)
    print(f"  {class_name:10s} → {len(wavs)} fichiers")

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f"\nTotal : {len(all_paths)} samples\n")

  talky      → 295 fichiers
  silence    → 135 fichiers
  unknown    → 326 fichiers

Total : 756 samples



## Pipeline d'entreainement dans une fonction à appeler plus tard

In [5]:
def run_single_experiment(run_idx):
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    
    print("=" * 55)
    print(f"  K-Fold stratifiée — {K_FOLDS} folds")
    print(f"  LR={LR}  |  Epochs={EPOCHS}  |  Batch={BATCH_SIZE}")
    print("=" * 55)
    
    for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(all_paths, all_labels)):
    
        print(f"\n[Fold {fold_idx+1}/{K_FOLDS}]")
    
        # ── Séparer train+val  /  test ────────────────────────────────────────
        X_trainval = all_paths[trainval_idx]
        y_trainval = all_labels[trainval_idx]
        X_test     = all_paths[test_idx]
        y_test     = all_labels[test_idx]
    
        # ── Découper val depuis train (20% de trainval = ~16% total) ─────────
        # On utilise un second StratifiedKFold à 5 folds pour prendre
        # exactement 1 bloc comme val → cohérent avec la logique k-fold
        inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED+fold_idx)
        train_sub_idx, val_sub_idx = next(inner_skf.split(X_trainval, y_trainval))
    
        X_train = X_trainval[train_sub_idx]
        y_train = y_trainval[train_sub_idx]
        X_val   = X_trainval[val_sub_idx]
        y_val   = y_trainval[val_sub_idx]
    
        print(f"  Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")
    

    class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
    )
    class_weight_dict = dict(enumerate(class_weights))


    ckpt_path = f"best_model_run{run_idx+1:02d}.keras"
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=0
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,                  
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=0
        ),
        tf.keras.callbacks.TensorBoard(log_dir="./logs")
    ]

    model = build_transfer_model(KERAS_PATH, num_classes=3)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    class_weight_dict = dict(enumerate(class_weights))


        # ── tf.data pipeline (identique au notebook) ─────────────────────────
    train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True)
    val_ds   = make_dataset(X_val,   y_val,   BATCH_SIZE, augment=False)
    test_ds  = make_dataset(X_test,  y_test,  BATCH_SIZE, augment=False)


    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=0                                    
    )


        # ── Modèle — reconstruit à chaque fold (poids frais) ─────────────────
    tf.keras.backend.clear_session()                         # ← libère la VRAM
    model = build_transfer_model(KERAS_PATH, num_classes=3)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )



    # --- Évaluation sur le meilleur modèle ---
    model = tf.keras.models.load_model(ckpt_path)
    loss, test_acc = model.evaluate(test_ds, verbose=0)

    # --- Infos supplémentaires ---
    best_epoch   = int(np.argmax(history.history["val_accuracy"])) + 1
    best_val_acc = float(np.max(history.history["val_accuracy"]))


    # Nettoyage du checkpoint pour ne pas saturer le disque
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)

    return {
        "test_accuracy": round(float(test_acc), 4),
        "val_accuracy":  round(best_val_acc, 4),
        "best_epoch":    best_epoch,
    }


## Fonction contanant la boucle for pour parcourir les 10 runs

In [6]:

import datetime


def run_all_experiments():
    n = N_RUNS
    all_results = []

    print("=" * 55)
    print(f"  KWS Transfer Learning — {n} runs")
    print(f"  LR={LR}  |  Epochs={EPOCHS}  |  Batch={BATCH_SIZE}")
    print("=" * 55)

    for i in range(n):
        seed = SEED
        print(f"\n[Run {i+1:02d}/{n}]  seed={seed} ...", flush=True)

        metrics = run_single_experiment(run_idx=i)
        metrics.update({"run": i+1, "seed": seed})
        all_results.append(metrics)

        print(f"test_acc={metrics['test_accuracy']:.4f}  "
              f"val_acc={metrics['val_accuracy']:.4f}  "
              f"best_epoch={metrics['best_epoch']}", flush=True)

    # --- Statistiques finales ---
    test_accs = np.array([r["test_accuracy"] for r in all_results])
    stats = {
        "mean":   round(float(np.mean(test_accs)),   4),
        "std":    round(float(np.std(test_accs)),    4),
        "median": round(float(np.median(test_accs)), 4),
        "min":    round(float(np.min(test_accs)),    4),
        "max":    round(float(np.max(test_accs)),    4),
    }

    print("\n" + "=" * 55)
    print("  RÉSULTATS FINAUX")
    print("=" * 55)
    for r in all_results:
        print(f"  Run {r['run']:02d} | seed={r['seed']:5d} | "
              f"test={r['test_accuracy']:.4f}  val={r['val_accuracy']:.4f}  "
              f"epoch={r['best_epoch']:3d}")
    print("-" * 55)
    print(f"  Mean   : {stats['mean']:.4f}")
    print(f"  Std    : {stats['std']:.4f}")
    print(f"  Median : {stats['median']:.4f}")
    print(f"  Min    : {stats['min']:.4f}  |  Max : {stats['max']:.4f}")
    print("=" * 55)

    # Conclusion
    m, s = stats["mean"], stats["std"]
    verdict = (
        "Excellente (≥90%)" if m >= 0.90 else
        "Bonne (≥80%)"      if m >= 0.80 else
        "Acceptable (≥70%)" if m >= 0.70 else
        "Insuffisante (<70%) — revoir le pipeline"
    )
    stability = "Stable" if s < 0.02 else f"Instable (std={s:.4f} > 0.02)"
    print(f"\n  Performance : {verdict}")
    print(f"  Stabilité   : {stability}")

    # Sauvegarde JSON
    output = {
        "timestamp":  datetime.datetime.now().isoformat(),
        "runs":       all_results,
        "statistics": stats,
        "conclusion": {"verdict": verdict, "stability": stability},
    }
    with open("results_multirun.json", "w") as f:
        json.dump(output, f, indent=2)
    print("\n  → Résultats sauvegardés dans results_multirun.json")


# ─────────────────────────────────────────────
if __name__ == "__main__":
    run_all_experiments()

  KWS Transfer Learning — 10 runs
  LR=0.001  |  Epochs=40  |  Batch=16

[Run 01/10]  seed=42 ...
  K-Fold stratifiée — 5 folds
  LR=0.001  |  Epochs=40  |  Batch=16

[Fold 1/5]
  Train : 483 | Val : 121 | Test : 152

[Fold 2/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 3/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 4/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 5/5]
  Train : 484 | Val : 121 | Test : 151


W0000 00:00:1781016074.454321  112129 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1781016074.491532  112073 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Restoring model weights from the end of the best epoch: 40.
test_acc=0.8675  val_acc=0.9091  best_epoch=12

[Run 02/10]  seed=42 ...
  K-Fold stratifiée — 5 folds
  LR=0.001  |  Epochs=40  |  Batch=16

[Fold 1/5]
  Train : 483 | Val : 121 | Test : 152

[Fold 2/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 3/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 4/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 5/5]
  Train : 484 | Val : 121 | Test : 151
Restoring model weights from the end of the best epoch: 39.
test_acc=0.8874  val_acc=0.9091  best_epoch=23

[Run 03/10]  seed=42 ...
  K-Fold stratifiée — 5 folds
  LR=0.001  |  Epochs=40  |  Batch=16

[Fold 1/5]
  Train : 483 | Val : 121 | Test : 152

[Fold 2/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 3/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 4/5]
  Train : 484 | Val : 121 | Test : 151

[Fold 5/5]
  Train : 484 | Val : 121 | Test : 151
Restoring model weights from the end of the best epoch: 40.
test_acc=0.8874  val_acc=0.9